# Structured Reasoning: Kaggle GPU Runner

This notebook clones the project from GitHub and runs a small GPU smoke test. Enable **GPU** in Notebook Settings and turn **Internet** on before running.

Start with 2 questions. If the run completes and the outputs look valid, run the 10-question pilot in the final cell.

In [ ]:
import torch, os, platform
print('Python:', platform.python_version())
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('GPU is not enabled. Open Notebook Settings → Accelerator → GPU, then restart the session.')

In [ ]:
# Clone the latest project files from GitHub
!git clone https://github.com/itsilsil/structured-reasoning-llm.git
%cd /kaggle/working/structured-reasoning-llm

In [ ]:
# Install project dependencies without reinstalling PyTorch (Kaggle provides it)
!pip install -q "transformers>=4.45.0" "accelerate>=0.34.0" "datasets>=3.0.0" "sentencepiece>=0.2.0" "pandas>=2.2.0" "tqdm>=4.66.0"

## 1. Smoke test — 2 questions

Run this first. It downloads the small Qwen model and the GSM8K test dataset. The first run may take a while.

In [ ]:
assert torch.cuda.is_available(), 'Enable GPU in Kaggle Notebook Settings before running the experiment.'
!python src/experiment.py --sample-size 2 --seed 42 --num-candidates 2 --max-new-tokens 160

In [ ]:
# Inspect the smoke-test summary
import pandas as pd
summary = pd.read_csv('results/summary.csv')
display(summary)
print('\nRun configuration:')
print(open('results/run_config.json', encoding='utf-8').read())

## 2. Full pilot — 10 questions

Only run this after the smoke test finishes successfully. This reruns the experiment with 10 questions.

In [ ]:
# Uncomment the next line to run the 10-question pilot:
# !python src/experiment.py --sample-size 10 --seed 42 --num-candidates 2 --max-new-tokens 160

# Then inspect the updated summary:
# display(pd.read_csv('results/summary.csv'))

## 3. Export results

Create a ZIP containing the CSV summary, raw JSONL records, and run configuration. Download it from the Kaggle output files after the cell runs.

In [ ]:
import zipfile, os
files = ['results/summary.csv', 'results/experiment_results.jsonl', 'results/run_config.json']
with zipfile.ZipFile('/kaggle/working/experiment_results.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    for path in files:
        if os.path.exists(path):
            z.write(path, arcname=os.path.basename(path))
print('Created /kaggle/working/experiment_results.zip')
print('Files:', [p for p in files if os.path.exists(p)])